# Flask: Serving ML Models with a Lightweight Web Framework

## What Is Flask?

Flask is the "just what you need" web framework.  
If FastAPI is a fully-equipped professional kitchen, Flask is a great home kitchen — everything essential, nothing unnecessary.  
You add tools (extensions) as you need them.

**Flask** is a lightweight WSGI web framework for Python:
- Minimal by design: no ORM, no form validation built-in (add them yourself)
- Mature and battle-tested (used by Pinterest, LinkedIn, Netflix)
- Excellent for ML model serving, REST APIs, and prototyping
- Synchronous (vs FastAPI's async) — simpler but handles fewer concurrent requests

## Resources

- **Docs**: [https://flask.palletsprojects.com/](https://flask.palletsprojects.com/)
- **GitHub**: [https://github.com/pallets/flask](https://github.com/pallets/flask)
- **YouTube — Flask Tutorial**: [https://www.youtube.com/watch?v=Z1RJmh_OqeA](https://www.youtube.com/watch?v=Z1RJmh_OqeA)

## Real-World Analogy

Think of Flask like a **food truck**:
- A restaurant (Django/Rails) has a full kitchen, waiters, a manager, a booking system — lots of built-in structure.
- A food truck (Flask) is just you, a grill, and a window. You decide everything: how to take orders, what to cook, how to serve.
- Perfect when you want **full control** without the overhead — exactly what you need for a lightweight ML model API.


## Installation

```bash
pip install flask
# For production: also install gunicorn
pip install gunicorn
```

In [ ]:
import json, pickle, time, logging
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

try:
    from flask import Flask, request, jsonify, make_response, g
    FLASK_AVAILABLE = True
    import flask
    print(f"Flask version: {flask.__version__}")
except ImportError:
    FLASK_AVAILABLE = False
    print("Flask not installed — simulated output shown. Install: pip install flask")

# Train a model to serve
np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train_s, y_train)
test_acc = accuracy_score(y_test, model.predict(scaler.transform(X_test)))
print(f"Model trained: {test_acc:.3f} test accuracy")

MODEL_VERSION = "1.0.0"
SERVER_START = time.time()

## Core Concept 1: Routes and the Request Object

Flask routes map URLs to Python functions.  
The `request` object contains everything about the incoming HTTP request.

In [ ]:
if FLASK_AVAILABLE:
    app = Flask(__name__)
    app.config['JSON_SORT_KEYS'] = False
    app.config['TESTING'] = True

    # ── Input validation helper (Flask has no Pydantic built-in) ─────────────
    def validate_features(data, n_expected=10):
        """Manual validation — Flask requires you to do this yourself."""
        errors = []
        if 'features' not in data:
            errors.append("'features' field is required")
        else:
            features = data['features']
            if not isinstance(features, list):
                errors.append("'features' must be a list")
            elif len(features) != n_expected:
                errors.append(f"'features' must have {n_expected} elements, got {len(features)}")
            else:
                for i, f in enumerate(features):
                    if not isinstance(f, (int, float)):
                        errors.append(f"features[{i}] must be a number, got {type(f).__name__}")
                    elif not np.isfinite(f):
                        errors.append(f"features[{i}] must be finite (no NaN/Inf)")
        return errors

    # ── Error handlers ────────────────────────────────────────────────────────
    @app.errorhandler(400)
    def bad_request(e):
        return jsonify({"error": "Bad Request", "message": str(e)}), 400

    @app.errorhandler(404)
    def not_found(e):
        return jsonify({"error": "Not Found", "message": str(e)}), 404

    @app.errorhandler(500)
    def server_error(e):
        return jsonify({"error": "Internal Server Error", "message": str(e)}), 500

    # ── Before/after request hooks ────────────────────────────────────────────
    @app.before_request
    def before_request():
        g.start_time = time.time()  # g = per-request global storage

    @app.after_request
    def after_request(response):
        # Add latency header to every response
        latency_ms = (time.time() - g.start_time) * 1000
        response.headers['X-Response-Time-ms'] = f"{latency_ms:.2f}"
        return response

    # ── Routes ────────────────────────────────────────────────────────────────
    @app.route('/health', methods=['GET'])
    def health():
        """Health check — used by load balancers."""
        return jsonify({
            "status": "healthy",
            "model_loaded": True,
            "version": MODEL_VERSION,
            "uptime_seconds": round(time.time() - SERVER_START, 1)
        })

    @app.route('/predict', methods=['POST'])
    def predict():
        """Single prediction endpoint."""
        # Parse JSON body
        if not request.is_json:
            return jsonify({"error": "Content-Type must be application/json"}), 400

        data = request.get_json()

        # Validate
        errors = validate_features(data, n_expected=10)
        if errors:
            return jsonify({"error": "Validation failed", "details": errors}), 422

        # Predict
        features = np.array(data['features']).reshape(1, -1)
        features_scaled = scaler.transform(features)
        prediction = int(model.predict(features_scaled)[0])
        proba = model.predict_proba(features_scaled)[0].tolist()

        return jsonify({
            "prediction": prediction,
            "confidence": round(max(proba), 4),
            "probabilities": [round(p, 4) for p in proba],
            "model_version": MODEL_VERSION,
        })

    @app.route('/predict/batch', methods=['POST'])
    def predict_batch():
        """Batch prediction endpoint."""
        data = request.get_json(force=True)
        instances = data.get('instances', [])

        if not instances or len(instances) > 1000:
            return jsonify({"error": "Provide 1-1000 instances"}), 400

        X_batch = np.array(instances)
        X_scaled = scaler.transform(X_batch)
        predictions = model.predict(X_scaled).tolist()
        probas = model.predict_proba(X_scaled).tolist()

        return jsonify({
            "predictions": predictions,
            "n_instances": len(predictions)
        })

    @app.route('/model-info', methods=['GET'])
    def model_info():
        return jsonify({
            "model_type": type(model).__name__,
            "n_estimators": model.n_estimators,
            "n_features": int(model.n_features_in_),
            "version": MODEL_VERSION
        })

    print("Flask app created with routes:")
    for rule in app.url_map.iter_rules():
        print(f"  {','.join(rule.methods - {'OPTIONS','HEAD'}):12s} {rule.rule}")

else:
    print("Flask app structure (simulated):")
    for m, p in [("GET", "/health"), ("POST", "/predict"),
                 ("POST", "/predict/batch"), ("GET", "/model-info")]:
        print(f"  {m:6s} {p}")

## Testing with Flask Test Client

In [ ]:
if FLASK_AVAILABLE:
    client = app.test_client()

    print("Test 1: Health check")
    resp = client.get('/health')
    print(f"  Status: {resp.status_code}")
    print(f"  Body: {json.dumps(resp.get_json(), indent=4)}")
    print()

    print("Test 2: Valid prediction")
    sample = X_test[0].tolist()
    resp = client.post('/predict',
                       data=json.dumps({'features': sample}),
                       content_type='application/json')
    print(f"  Status: {resp.status_code}")
    print(f"  Body: {json.dumps(resp.get_json(), indent=4)}")
    print()

    print("Test 3: Wrong feature count → 422")
    resp = client.post('/predict',
                       data=json.dumps({'features': [1.0, 2.0]}),
                       content_type='application/json')
    print(f"  Status: {resp.status_code}")
    print(f"  Body: {resp.get_json()}")
    print()

    print("Test 4: Batch prediction (5 samples)")
    batch = X_test[:5].tolist()
    resp = client.post('/predict/batch',
                       data=json.dumps({'instances': batch}),
                       content_type='application/json')
    result = resp.get_json()
    print(f"  Status: {resp.status_code}")
    print(f"  Predictions: {result['predictions']}")

else:
    print("Simulated Flask test output:")
    print()
    print("Test 1 — GET /health → 200")
    print('  {"status": "healthy", "model_loaded": true, "version": "1.0.0"}')
    print()
    print("Test 2 — POST /predict (valid) → 200")
    print('  {"prediction": 1, "confidence": 0.84, "probabilities": [0.16, 0.84]}')
    print()
    print("Test 3 — POST /predict (wrong size) → 422")
    print('  {"error": "Validation failed", "details": ["features must have 10 elements, got 2"]}')
    print()
    print("Test 4 — POST /predict/batch → 200")
    print('  {"predictions": [1, 0, 1, 1, 0], "n_instances": 5}')

## Blueprints — Organizing Large Flask Apps

For large apps, organize routes into **Blueprints** (like modules for routes).

In [ ]:
print("Flask Blueprints — organizing a large ML serving app:")
print()
print("""
# File structure:
#   app/
#     __init__.py      # create_app() factory
#     models/
#       predict.py     # /predict blueprint
#     monitoring/
#       health.py      # /health, /metrics blueprint
#     auth/
#       middleware.py  # API key verification

# app/models/predict.py
from flask import Blueprint, request, jsonify

predict_bp = Blueprint('predict', __name__, url_prefix='/v1')

@predict_bp.route('/predict', methods=['POST'])
def predict():
    ...

# app/__init__.py
def create_app(config=None):
    app = Flask(__name__)
    from .models.predict import predict_bp
    from .monitoring.health import health_bp
    app.register_blueprint(predict_bp)
    app.register_blueprint(health_bp)
    return app

# All routes now accessible:
# POST /v1/predict
# GET  /health
""")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Using `app.run()` in production | Single worker, slow | Use Gunicorn: `gunicorn -w 4 main:app` |
| Global model variable | State issues in multi-threaded | Use `g` or app-level singleton; load at import time |
| Not using `request.get_json(force=True)` | None if wrong Content-Type | Use `force=True` or check `request.is_json` |
| Returning Python dicts directly | `TypeError` | Always use `jsonify()` |
| No error handlers | Ugly HTML error pages for JSON clients | Register `@app.errorhandler(400)` etc. |
| debug=True in production | Security risk (interactive debugger) | Set `debug=False` and use env vars |

## Interview Questions and Answers

## Interview Questions & Answers

---

**Q1: What is WSGI and why does Flask need Gunicorn in production?**

A: WSGI (Web Server Gateway Interface) is a Python standard (PEP 3333) defining how web servers pass HTTP requests to Python applications. Flask implements the WSGI interface — it is a *WSGI application*, not a web server itself. `flask run` uses a single-threaded development server (Werkzeug) that handles one request at a time — fine for testing, catastrophic for production (one slow request blocks everything). Gunicorn is a production-grade WSGI *server* that spawns multiple worker processes, each handling requests independently. Rule: always use `gunicorn -w 4 app:app` in production, never `flask run`.

---

**Q2: How does Flask differ from FastAPI? When would you choose Flask?**

A: **FastAPI** has automatic data validation (Pydantic), type hints, async support, and auto-generated OpenAPI docs — ideal for new ML APIs. **Flask** is simpler, has a massive ecosystem (10+ years of extensions), and is still the right choice when: (1) you are maintaining an existing Flask codebase; (2) you need a specific Flask extension (Flask-SQLAlchemy, Flask-Login); (3) the team already knows Flask; (4) you need total control over request/response handling. For a new ML service with no legacy constraints, choose FastAPI. For extending an existing web app, Flask is perfectly fine.

---

**Q3: What is the Global Interpreter Lock (GIL) and how does it affect Flask serving?**

A: The GIL is a mutex in CPython that allows only one thread to execute Python bytecode at a time. This means Flask with threading cannot truly parallelise CPU-bound work (model inference) across cores. Solutions: (1) **Gunicorn + multiple processes** (`-w N`) — each process has its own GIL and Python interpreter, true parallelism; (2) **Async framework** (FastAPI + uvicorn) — sidesteps GIL for I/O-bound work; (3) **Move inference to a separate service** (e.g., Triton Inference Server) that is called over gRPC. For ML serving, multi-process Gunicorn is the standard Flask deployment pattern.

---

**Q4: What is a Flask Blueprint and when should you use one?**

A: A Blueprint is a reusable collection of routes, templates, and static files that can be registered on a Flask application. Use Blueprints when: (1) your app has more than 2-3 functional areas (auth, predictions, admin); (2) you want to split code across multiple files; (3) you want to test route groups independently. Structure: `auth_bp = Blueprint("auth", __name__)` → define routes on `auth_bp` → `app.register_blueprint(auth_bp, url_prefix="/auth")`. Without Blueprints, large Flask apps become one enormous file — Blueprints impose the same clean separation that Django apps provide automatically.

---

**Q5: How do you handle model loading efficiently in a Flask ML API?**

A: Never load the model inside the route handler — that would load it on every request (seconds of latency). Three patterns: (1) **Module-level global** — `model = pickle.load(...)` at the top of `app.py` — loaded once at startup, shared across all requests (thread-safe for read-only inference); (2) **`@app.before_first_request`** / `with app.app_context()` — deferred loading; (3) **Flask-Caching or Redis** — for models that need periodic refresh. For Gunicorn multi-process, each worker loads its own copy — for very large models, use a shared memory approach or a dedicated model server (Triton, TorchServe).

---

**Q6: What is the difference between `app.run(debug=True)` and a production deployment?**

A: `debug=True` enables: (1) **Auto-reload** — code changes restart the server automatically; (2) **Interactive debugger** — if an exception occurs, a web-based debugger appears in the browser; (3) **Detailed error pages**. In production, all three are dangerous: the interactive debugger exposes an executable Python console to anyone who can reach the error page — this is a critical security vulnerability (arbitrary code execution). Always set `debug=False` (or remove it), use environment variables for configuration, and serve behind a reverse proxy (Nginx) that hides stack traces from end users.


In [ ]:
qa = [
    {"q": "What is WSGI and how does Flask use it?",
     "a": """WSGI (Web Server Gateway Interface) is a standard interface between web servers and Python web apps.

How it works:
1. Nginx/Apache receives HTTP request
2. Passes it to WSGI server (Gunicorn, uWSGI)
3. WSGI server calls your Flask app as a Python callable
4. Flask processes request, returns response
5. WSGI server sends response back through Nginx

WSGI is synchronous: one thread handles one request at a time.
Gunicorn with 4 workers (-w 4) = 4 concurrent requests max.

vs ASGI (FastAPI):
ASGI is async: one thread handles many requests via async/await.
Better for I/O-bound work; WSGI is fine for CPU-bound ML inference.

For ML serving: WSGI is usually fine. Model inference is CPU-bound,
so async doesn't help much. Use enough Gunicorn workers instead."""},

    {"q": "Flask vs FastAPI — when would you still choose Flask?",
     "a": """Choose Flask when:
1. Existing Flask codebase: not worth migrating for the sake of it
2. Team is more familiar with Flask
3. Many Flask extensions needed (Flask-SQLAlchemy, Flask-Login, etc.)
4. Simple API where FastAPI's features aren't needed
5. Tutorial/learning project where simplicity helps

In practice: Flask and FastAPI are both excellent for ML serving.
The main concrete advantages of FastAPI are:
- Automatic Pydantic validation (vs manual validation in Flask)
- Auto-generated OpenAPI docs
- Better async support

For a new ML API: FastAPI is the better choice.
For maintaining an existing Flask app: keep Flask."""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Concept | Flask API |
|---------|----------|
| Create app | `app = Flask(__name__)` |
| Define route | `@app.route('/path', methods=['GET','POST'])` |
| Read JSON body | `request.get_json()` |
| Return JSON | `jsonify({...})` |
| Error handler | `@app.errorhandler(404)` |
| Before request | `@app.before_request` |
| Blueprints | `Blueprint('name', __name__, url_prefix='/v1')` |
| Testing | `app.test_client().post('/path', data=json, content_type='application/json')` |
| Production | `gunicorn -w 4 -b 0.0.0.0:5000 main:app` |

### Next Steps
1. **Flask quickstart**: [https://flask.palletsprojects.com/quickstart/](https://flask.palletsprojects.com/quickstart/)
2. **Flask + ML**: [https://www.youtube.com/watch?v=UbCWoMf80PY](https://www.youtube.com/watch?v=UbCWoMf80PY)
3. **Next**: Learn BentoML for ML-specific packaging with automatic containerization